In [0]:
import json
kafka_connections_json = dbutils.secrets.get(scope = 'finguard-scope', key = "kafka_connection_details")
kafka_config = json.loads(kafka_connections_json)
bootstrap_server = kafka_config['bootstrap_server']
api_key = kafka_config['api_key']
api_secret = kafka_config['api_secret']
topic = kafka_config['topic']


In [0]:
jaas_config = "kafkashaded.org.apache.kafka.common.security.plain.PlainLoginModule required username=\"{}\" password=\"{}\";".format(api_key,api_secret)

In [0]:
sample_batch = (spark.read.format("kafka")
                .option("kafka.bootstrap.servers",bootstrap_server)
                .option("subscribe",topic)
                .option("kafka.security.protocol","SASL_SSL")
                .option("kafka.sasl.jaas.config",jaas_config)
                .option("kafka.sasl.mechanism","PLAIN")
                .option("startingOffsets","earliest")
                .load()
        
                )

In [0]:
sample_batch.count()

In [0]:
display(sample_batch)

In [0]:
from pyspark.sql.functions import col

parsed_batch = sample_batch.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

In [0]:
display(parsed_batch)

In [0]:
parsed_batch.write.saveAsTable("financial_guard.bronze.transactions_batch_test")

In [0]:
streaming_df = (spark.readStream.format("kafka")
                .option("kafka.bootstrap.servers",bootstrap_server)
                .option("subscribe",topic)
                .option("kafka.security.protocol","SASL_SSL")
                .option("kafka.sasl.jaas.config",jaas_config)
                .option("kafka.sasl.mechanism","PLAIN")
                .option("startingOffsets","earliest")
                .load()
        
                )

In [0]:
from pyspark.sql.functions import col

parsed_streaming_df = streaming_df.select(
col("key").cast("string"),
col("value").cast("string"),
col("topic"),
col("partition"),
col("offset"),
col("timestamp"),
col("timestampType")
)

In [0]:
streaming_query = (parsed_streaming_df.writeStream.format("delta")
.outputMode("Append")
.option("checkpointLocation","/Volumes/financial_gaurd/source/transactions/checkpoint/")
.trigger(availableNow=True)
.toTable("financial_guard.bronze.transactions_streaming_test")
)

print("Query Started with Query Id: ", streaming_query.id)

In [0]:
%sql
SELECT * FROM financial_gaurd.bronze.transactions_streaming_test

In [0]:
%sql
SELECT COUNT(*) FROM financial_guard.silver.transactions